# Dataset downloader for MBMF Q1 v8

This notebook downloads/extracts raw datasets only. **All preprocessing/splitting remains inside `mbmf_q1_infrastructure_v8.py`.**

In [ ]:
from pathlib import Path
import hashlib, json, zipfile, time
import requests

DATA_ROOT=Path("./datasets").resolve()
CACHE=DATA_ROOT/"_downloads"
DATA_ROOT.mkdir(parents=True,exist_ok=True); CACHE.mkdir(parents=True,exist_ok=True)
print(DATA_ROOT)

In [ ]:
def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''): h.update(b)
    return h.hexdigest()

def download(url,target,min_bytes=1024,retries=3):
    target=Path(target); target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.stat().st_size>=min_bytes:
        print('[cached]',target); return target
    tmp=target.with_suffix(target.suffix+'.part'); last=None
    for n in range(1,retries+1):
        try:
            print('[download]',url)
            with requests.get(url,stream=True,timeout=120,headers={'User-Agent':'Mozilla/5.0 Q1-research'}) as r:
                r.raise_for_status()
                with open(tmp,'wb') as f:
                    for chunk in r.iter_content(1<<20):
                        if chunk: f.write(chunk)
            if tmp.stat().st_size<min_bytes: raise RuntimeError('download too small')
            tmp.replace(target); return target
        except Exception as e:
            last=e; time.sleep(2*n)
    raise RuntimeError(url) from last

def extract(z,dst):
    dst=Path(dst); dst.mkdir(parents=True,exist_ok=True); base=dst.resolve()
    with zipfile.ZipFile(z) as a:
        for m in a.infolist():
            out=(dst/m.filename).resolve()
            if out!=base and base not in out.parents: raise RuntimeError('unsafe zip member')
        a.extractall(dst)

def first(urls,target,min_bytes):
    errors=[]
    for u in urls:
        try:return download(u,target,min_bytes)
        except Exception as e:errors.append((u,repr(e)))
    raise RuntimeError(errors)


## MovieLens 100K / 1M / 10M

In [ ]:
urls={
"ml-100k":"https://files.grouplens.org/datasets/movielens/ml-100k.zip",
"ml-1m":"https://files.grouplens.org/datasets/movielens/ml-1m.zip",
"ml-10m":"https://files.grouplens.org/datasets/movielens/ml-10m.zip",
}
for name,url in urls.items():
    z=download(url,CACHE/f"{name}.zip",1_000_000); extract(z,DATA_ROOT)

## Jester Dataset 1

In [ ]:
jdir=DATA_ROOT/"jester"; jdir.mkdir(exist_ok=True)
for arc in ("jester_dataset_1_1.zip","jester_dataset_1_2.zip","jester_dataset_1_3.zip"):
    z=download("https://eigentaste.berkeley.edu/dataset/"+arc,CACHE/arc,500_000); extract(z,jdir)

## Book-Crossing

In [ ]:
bdir=DATA_ROOT/"bookcrossing"; bdir.mkdir(exist_ok=True)
z=first([
"https://cdn.freecodecamp.org/project-data/books/book-crossings.zip",
"http://www2.informatik.uni-freiburg.de/~cziegler/BX/BX-CSV-Dump.zip",
],CACHE/"book-crossings.zip",10_000_000)
extract(z,bdir)

## Verify and write raw-download manifest

In [ ]:
expected=[
DATA_ROOT/"ml-100k"/"u.data", DATA_ROOT/"ml-1m"/"ratings.dat", DATA_ROOT/"ml-10M100K"/"ratings.dat",
DATA_ROOT/"jester"/"jester-data-1.xls", DATA_ROOT/"jester"/"jester-data-2.xls", DATA_ROOT/"jester"/"jester-data-3.xls",
DATA_ROOT/"bookcrossing"/"BX-Book-Ratings.csv",
]
missing=[str(p) for p in expected if not p.exists()]
if missing: raise FileNotFoundError(missing)
manifest=[]
for p in sorted(CACHE.glob('*')):
    if p.is_file(): manifest.append({'file':str(p.resolve()),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
(DATA_ROOT/'dataset_download_manifest_v8.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
print('verified',len(expected),'key raw files')